#Evaluating Different Approaches to Improve SVM Accuracy

Support Vector Machines (SVM) are powerful classifiers, but their performance can be significantly influenced by various factors, including feature representation, hyperparameter tuning, and data preprocessing techniques. In this evaluation, we explore different strategies to improve the accuracy of SVM for relation classification.

Please follow the guidelines provided in each description of the approaches. Each approach should be execute and evaluated separately according to the instructions provided.

# Import required libraries

In [1]:
import numpy as np  # For numerical operations
import pandas as pd  # For handling datasets in tabular format
import spacy  # NLP library for tokenization and dependency parsing
import nltk  # Natural Language Toolkit for POS tagging and text processing
from nltk import word_tokenize, pos_tag, sent_tokenize  # Tokenization and Part-of-Speech (POS) tagging

# Import machine learning and preprocessing libraries
from sklearn.pipeline import Pipeline  # For creating a pipeline of processing steps
from sklearn.svm import SVC  # Support Vector Machine classifier
from sklearn.feature_extraction.text import TfidfVectorizer  # Convert text into numerical features
from sklearn.preprocessing import LabelEncoder  # Encode labels into numeric values
from sklearn.model_selection import train_test_split, GridSearchCV  # Split dataset and tune hyperparameters
from sklearn.metrics import classification_report  # Evaluate model performance

# Install and import Hugging Face datasets library (used for loading datasets)
!pip install datasets
from datasets import load_dataset  # Load datasets from Hugging Face

# Importing Word2Vec model from Gensim
from gensim.models import Word2Vec

# Download necessary NLTK resources:
# 1. "averaged_perceptron_tagger_eng" for POS tagging
# 2. "punkt_tab" for sentence tokenizer
# 3. "punkt" for word tokenizer
nltk.download('averaged_perceptron_tagger_eng')
nltk.download('punkt_tab')
nltk.download('punkt')

# Importing DictVectorizer from sklearn to convert dictionaries into numeric feature vectors
from sklearn.feature_extraction import DictVectorizer

# Load the pre-trained SpaCy model for English for dependency parsing and NLP tasks
nlp = spacy.load('en_core_web_sm')

# Import hstack from scipy to combine sparse matrices
from scipy.sparse import hstack

# Import classification_report and confusion_matrix from sklearn for evaluating model performance
from sklearn.metrics import classification_report, confusion_matrix

# Import Gensim API to download and load pre-trained word embeddings like GloVe
import gensim.downloader as api
glove_model = api.load("glove-wiki-gigaword-100")  # Pretrained 100D vectors

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 485.4/485.4 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 9.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.8/194.8 kB 12.8 MB/s eta 0:00:00


[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger_eng.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


[==================================================] 100.0% 128.1/128.1MB downloaded


# Load Dataset

In [2]:
# Load the dataset
ds = load_dataset("SemEvalWorkshop/sem_eval_2010_task_8")

# Check dataset structure
print(ds)

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/8.23k [00:00<?, ?B/s]

train-00000-of-00001.parquet:   0%|          | 0.00/673k [00:00<?, ?B/s]

test-00000-of-00001.parquet:   0%|          | 0.00/231k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/8000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/2717 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['sentence', 'relation'],
        num_rows: 8000
    })
    test: Dataset({
        features: ['sentence', 'relation'],
        num_rows: 2717
    })
})


# Convert to Pandas Dataframe

In [3]:
# Convert Hugging Face dataset to Pandas DataFrame
df_train = pd.DataFrame(ds["train"])
df_test = pd.DataFrame(ds["test"])

# Check the first few rows
print(df_train.head())
print(df_train.shape)

                                            sentence  relation
0  The system as described above has its greatest...         3
1  The <e1>child</e1> was carefully wrapped and b...        18
2  The <e1>author</e1> of a keygen uses a <e2>dis...        11
3  A misty <e1>ridge</e1> uprises from the <e2>su...        18
4  The <e1>student</e1> <e2>association</e2> is t...        12
(8000, 2)


# Preprocessing Approach 1: TFIDF (Accuracy 59% best param(C=10))

This section performs feature extraction using TF-IDF Vectorization to convert textual data into numerical representations. TF-IDF (Term Frequency-Inverse Document Frequency) helps capture the importance of words while reducing the impact of commonly used words. This transformation is essential for training the SVM model on meaningful features.



In [4]:
# Initialize TF-IDF Vectorizer
vectorizer = TfidfVectorizer(
    lowercase=True,       # Convert text to lowercase for uniformity
    stop_words='english', # Remove common English words that don't add much meaning
    ngram_range=(1,2)     # Consider both single words (unigrams) and word pairs (bigrams)
)

# Transform training data into numerical feature vectors
X_train = vectorizer.fit_transform(df_train['sentence'])  # Learn vocabulary & transform training sentences
y_train = df_train['relation']  # Extract target labels for training

# Transform test data using the same vectorizer
X_test = vectorizer.transform(df_test['sentence'])  # Convert test sentences into numerical features
y_test = df_test['relation']  # Extract target labels for evaluation


# Preprocessing Approach 2: Word Embedding (Accuracy 36%)

In this section, we utilize Word2Vec embeddings to represent sentences as numerical vectors. Word2Vec learns word representations based on context, allowing us to capture semantic similarities between words. By averaging word embeddings, we generate sentence-level representations, which serve as features for training the SVM model.

In [5]:
# Tokenize sentences in the training set
df_train['tokenized'] = df_train['sentence'].apply(word_tokenize)  # Split sentences into words

# Train Word2Vec model ONLY on the training data
word2vec_model = Word2Vec(
    sentences=df_train['tokenized'],  # Train on tokenized sentences
    vector_size=100,  # Set embedding size to 100 dimensions
    window=5,         # Consider 5 words before and after the target word
    min_count=1       # Include all words appearing at least once
)

# Function to convert a sentence into a vector by averaging word embeddings
def get_sentence_vector(sentence, model):
    words = word_tokenize(sentence)  # Tokenize sentence into words
    vectors = [model.wv[word] for word in words if word in model.wv]  # Extract word embeddings
    return sum(vectors) / len(vectors) if vectors else np.zeros(model.vector_size)  # Compute average vector

# Convert training sentences into vectors
X_train = np.vstack(df_train['sentence'].apply(lambda s: get_sentence_vector(s, word2vec_model)).values)

# Extract target labels for training
y_train = df_train['relation']

# Tokenize sentences in the test set
df_test['tokenized'] = df_test['sentence'].apply(word_tokenize)  # Split sentences into words

# Convert test sentences into vectors using the trained Word2Vec model
X_test = np.vstack(df_test['sentence'].apply(lambda s: get_sentence_vector(s, word2vec_model)).values)

# Extract target labels for evaluation
y_test = df_test['relation']

# Preprocessing Approach 3: TFIDF with POS Tagging (Accuracy 64% best param(C=10))

In this section, we first apply Part-of-Speech (POS) tagging to the sentences in the training and test datasets to capture grammatical structures. The sentences are then transformed into TF-IDF features to be used as input for machine learning models. This representation highlights important words and their relative importance within the sentences.

In [6]:
# Function to get POS tags as a string
def get_pos_tags(sentence):
    tokens = word_tokenize(sentence)  # Tokenize the sentence into words
    pos_tags = pos_tag(tokens)  # Apply POS tagging
    return " ".join([f"{word}_{tag}" for word, tag in pos_tags])  # Return POS-tagged words as a string

# Apply POS tagging to all sentences in the training set
df_train['pos_sentence'] = df_train['sentence'].apply(get_pos_tags)  # Add POS-tagged sentences to the train dataset

# Apply POS tagging to all sentences in the test set
df_test['pos_sentence'] = df_test['sentence'].apply(get_pos_tags)  # Add POS-tagged sentences to the test dataset

# Initialize the TF-IDF vectorizer
vectorizer = TfidfVectorizer(lowercase=True, stop_words='english', ngram_range=(1, 2))  # Using 1-gram and 2-gram

# Convert training sentences to numerical features using TF-IDF
X_train = vectorizer.fit_transform(df_train['pos_sentence'])  # Apply vectorization to the train data
y_train = df_train['relation']  # Extract target labels for training

# Convert test sentences to numerical features using the same vectorizer
X_test = vectorizer.transform(df_test['pos_sentence'])  # Apply vectorization to the test data
y_test = df_test['relation']  # Extract target labels for testing

# Approach 4: using Glove Word Embedding (Acurracy 51% best param(C=1))

In this part of the code, we convert sentences into GloVe word embeddings by tokenizing each sentence and retrieving the corresponding GloVe vectors for each word. The resulting word vectors are averaged to produce a single vector representation of the sentence. These sentence vectors are then used as input features for machine learning models.


In [7]:
# Function to get GloVe vector for a sentence by averaging the word embeddings
def get_glove_vector(sentence):
    words = word_tokenize(sentence)  # Tokenize the sentence into words
    vectors = [glove_model[word] for word in words if word in glove_model]  # Retrieve GloVe vectors for each word
    return sum(vectors) / len(vectors) if vectors else np.zeros(100)  # Return the average vector of words, or zeros if no words found in GloVe

# Convert training sentences to GloVe vectors
X_train = np.vstack(df_train['sentence'].apply(get_glove_vector).values)  # Apply the GloVe conversion function to all sentences in the train set

# Convert test sentences to GloVe vectors
X_test = np.vstack(df_test['sentence'].apply(get_glove_vector).values)  # Apply the GloVe conversion function to all sentences in the test set

# Approach 5: using Glove with POS Tagging (Accracy 43%)

This code processes sentences by tokenizing them and applying Part-of-Speech (POS) tagging to extract only the nouns, verbs, and adjectives. These filtered words are then used to generate GloVe word embeddings. The vectors corresponding to these words are averaged to create a single vector representation for each sentence. This approach ensures that only the most informative parts of the sentence, based on POS tags, contribute to the sentence's embedding.

In [8]:
# Function to get GloVe vector for a sentence by filtering nouns, verbs, and adjectives
def get_glove_vector(sentence):
    words = word_tokenize(sentence)  # Tokenize the sentence into words
    pos_tags = pos_tag(words)  # Get POS tags for each word in the sentence
    filtered_words = [word for word, tag in pos_tags if tag.startswith(('N', 'V', 'J'))]  # Keep only nouns, verbs, adjectives

    vectors = [glove_model[word] for word in filtered_words if word in glove_model]  # Retrieve GloVe vectors for filtered words
    return sum(vectors) / len(vectors) if vectors else np.zeros(100)  # Return the average vector of filtered words, or a zero vector if no match

# Convert training sentences to GloVe vectors
X_train = np.vstack(df_train['sentence'].apply(get_glove_vector).values)  # Apply the GloVe conversion function to all sentences in the train set

# Convert test sentences to GloVe vectors
X_test = np.vstack(df_test['sentence'].apply(get_glove_vector).values)  # Apply the GloVe conversion function to all sentences in the test set

# Target labels
y_train = df_train['relation']  # Target labels for training
y_test = df_test['relation']  # Target labels for testing

# Approach 6: Syntactic and Dependency Features (Accuracy 24%)

This code demonstrates how to extract syntactic features from sentences using spaCy's dependency parsing and how to use these features to train a Linear SVM model for text classification. The syntactic features include the word itself, its Part-of-Speech (POS) tag, dependency label, the head word, POS of the head word, stopword status, and the distance to the head word. These features are then converted to numeric format using DictVectorizer, and a Linear SVM is trained to classify sentences into different relation categories.

In [9]:
# Function to extract syntactic features from a sentence
def extract_syntactic_features(sentence):
    doc = nlp(sentence)  # Parse sentence with spaCy
    features = []

    for token in doc:
        # Feature: word, its POS tag, and its dependency label
        features.append({
            'word': token.text,  # Token's text
            'pos': token.pos_,  # Part-of-speech tag
            'dep': token.dep_,  # Dependency relation
            'head_word': token.head.text,  # Head word of the token
            'head_pos': token.head.pos_,  # Part-of-speech of the head word
            'is_stop': token.is_stop,  # Whether the token is a stopword
            'dep_distance': token.head.i - token.i  # Distance from the head word
        })

    return features

# Extract features for all sentences in df_train and df_test
X_train = [extract_syntactic_features(sentence) for sentence in df_train['sentence']]  # Features for train set
X_test = [extract_syntactic_features(sentence) for sentence in df_test['sentence']]  # Features for test set

# Use DictVectorizer to convert features to numeric format (sparse matrix)
vectorizer = DictVectorizer(sparse=True)  # Initialize DictVectorizer
X_train = vectorizer.fit_transform([f[0] for f in X_train])  # Convert train features to sparse matrix
X_test = vectorizer.transform([f[0] for f in X_test])  # Convert test features to sparse matrix

# Use the 'relation' column as target labels
y_train = df_train['relation']  # Target labels for training
y_test = df_test['relation']  # Target labels for testing

# Approach 7: TFIDF + POS Tagging + Dependency features (Accuracy 63%)

This code focuses on extracting features from sentences using spaCy's dependency parsing and then applying TF-IDF vectorization to these features for use in machine learning models. The features include the word itself, its Part-of-Speech (POS) tag, dependency relation, and its head word. These features are then represented numerically using the TF-IDF vectorizer, which helps the model to understand the structure of the sentence. The resulting feature set is used to predict sentence relations using machine learning models like SVM.

In [10]:
def get_dependency_features(sentence):
    doc = nlp(sentence)  # Parse the sentence using spaCy
    features = []

    # Loop through the tokens (words) in the sentence
    for token in doc:
        # Feature: word, its POS tag, its dependency relation, and the head word of the token
        features.append(f"{token.text}_{token.pos_}_{token.dep_}_{token.head.text}")

    # Return a string of all the features
    return " ".join(features)

# Apply dependency feature extraction to the train and test data
df_train['dependency_sentence'] = df_train['sentence'].apply(get_dependency_features)  # Train set
df_test['dependency_sentence'] = df_test['sentence'].apply(get_dependency_features)  # Test set

# Set the target labels for the classification task
y_train = df_train['relation']
y_test = df_test['relation']

# Apply TF-IDF Vectorization on the extracted syntactic features
vectorizer = TfidfVectorizer(lowercase=True, stop_words='english', ngram_range=(1,2))  # Consider unigrams and bigrams
X_train = vectorizer.fit_transform(df_train['dependency_sentence'])  # Train set to numeric format
X_test = vectorizer.transform(df_test['dependency_sentence'])  # Test set to numeric format

# Approach 8: (Hybrid = GLove + Approach 7) (Accuracy 67%) Best param(C=10)

In this code, we combine Approach 7 with pretrained GloVe embeddings to enhance the feature representation for sentence classification. The GloVe model is used to obtain dense word embeddings that capture semantic meanings, while TF-IDF represents the importance of words in relation to the entire corpus. By combining both, we create a richer feature set that can potentially improve the performance of the machine learning model. The SVM classifier is then trained on this combined feature set to predict sentence relations.

Note: Run approach 7 before this!!

In [11]:
def get_glove_vector(sentence):
    """
    This function generates a GloVe vector for a given sentence.
    It tokenizes the sentence, looks up the corresponding GloVe vectors for the words,
    and returns the average of the vectors. If a word does not exist in the GloVe model,
    it is ignored.
    """
    words = word_tokenize(sentence)  # Tokenize the sentence into words
    vectors = [glove_model[word] for word in words if word in glove_model]  # Get vectors for words in the sentence
    return sum(vectors) / len(vectors) if vectors else np.zeros(100)  # Return the average vector or a zero vector if no valid words

# Generate GloVe embeddings for all sentences in the training set
X_train_glove = np.vstack(df_train['sentence'].apply(get_glove_vector).values)  # Convert sentences to vectors
X_test_glove = np.vstack(df_test['sentence'].apply(get_glove_vector).values)  # Convert sentences to vectors

# Combine TF-IDF (sparse) features with GloVe (dense) embeddings into a single feature matrix
X_train_combined = hstack((X_train, X_train_glove))  # Train set: merge sparse TF-IDF and dense GloVe
X_test_combined = hstack((X_test, X_test_glove))  # Test set: merge sparse TF-IDF and dense GloVe

# Train SVM with the combined feature set
svm_model = SVC(kernel='linear', C=10)  # Initialize the SVM with a linear kernel
svm_model.fit(X_train_combined, y_train)  # Train the SVM model
predictions = svm_model.predict(X_test_combined)  # Predict the labels for the test set

# Snippet for the Grid Search Algorithm

In this code, GridSearchCV is used to perform hyperparameter tuning for an SVM model. Specifically, the regularization parameter C is tested over a range of values to identify the optimal setting that yields the best model performance. Grid search performs cross-validation (here, 5-fold) to evaluate the model's accuracy for each combination of parameters. This helps in selecting the best hyperparameter configuration for better generalization.

In [12]:
# Define the parameter grid for hyperparameter tuning
param_grid = {'C': [0.1, 1, 10, 100]}  # Test different values for the regularization parameter C

# Perform grid search again, this time using only the TF-IDF features
grid_search = GridSearchCV(SVC(kernel='linear'), param_grid, cv=5)
grid_search.fit(X_train, y_train)  # Train the grid search model on the TF-IDF feature set

# Output the best parameters from the grid search
print("Best Parameters:", grid_search.best_params_)

/usr/local/lib/python3.11/dist-packages/sklearn/model_selection/_split.py:805: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=5.
  warnings.warn(


Best Parameters: {'C': 10}


#Train SVM Model

This code trains a Linear Support Vector Machine (SVM) model on the training data and then makes predictions on the test set. The SVM model is used for text classification, and it works by finding the optimal hyperplane that separates different classes in the feature space. After training, the model's predictions on the test set are displayed.

Note: Approach 8 has it's own SVM Training snippet in the code cell as it uses different variable set for training.

In [13]:
# Train Linear SVM Model
svm_model = SVC(kernel='linear', C=10)  # Linear kernel for text classification
svm_model.fit(X_train, y_train)  # Train the SVM model on the training data

# Make Predictions
predictions = svm_model.predict(X_test)  # Predict labels for the test set
print("Predictions:", predictions)  # Print the predicted labels

Predictions: [18 18 18 ...  9 14 18]


# Classification Report

This code evaluates the performance of the trained SVM model by calculating the classification report and confusion matrix. These metrics help understand how well the model is performing across different classes by measuring precision, recall, F1-score, and support.

In [14]:
# Classification Report (includes Precision, Recall, F1-score)
print("Classification Report:\n", classification_report(y_test, predictions))

Classification Report:
               precision    recall  f1-score   support

           0       0.95      0.72      0.82       134
           1       0.83      0.81      0.82       194
           2       0.56      0.70      0.62       162
           3       0.68      0.47      0.56       150
           4       0.73      0.86      0.79       153
           5       0.87      0.69      0.77        39
           6       0.82      0.89      0.86       291
           7       0.00      0.00      0.00         1
           8       0.81      0.75      0.78       211
           9       0.43      0.55      0.49        47
          10       0.67      0.18      0.29        22
          11       0.63      0.51      0.57       134
          12       0.73      0.25      0.37        32
          13       0.60      0.89      0.72       201
          14       0.73      0.54      0.62       210
          15       0.84      0.31      0.46        51
          16       0.66      0.52      0.58       108
   

/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
